# TLC — AttentiveFP



In [ ]:
%pip install -q rdkit torch-geometric pandas ipywidgets


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
MODEL_NAME = 'AttentiveFP'
# Set this path to your saved checkpoint in Google Drive.
CHECKPOINT = Path('/content/drive/MyDrive/TLC/Rf_GNN_random_outputs/AttentiveFP/20261115T135528207396Z/best_model.pt')


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem
from torch_geometric.data import Data, Batch
ATOM_TYPES = [
    1, 5, 6, 7, 8, 9, 14, 15, 16, 17,
    35, 53
]

HYBRIDIZATIONS = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2
]

BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC
]

def one_hot_unknown(value, choices):
    return [float(value == c) for c in choices] + [
        float(value not in choices)
    ]

def atom_features(atom):
    features = []
    features += one_hot_unknown(
        atom.GetAtomicNum(), ATOM_TYPES
    )
    features += one_hot_unknown(
        atom.GetHybridization(), HYBRIDIZATIONS
    )
    features += [
        atom.GetDegree() / 6.0,
        atom.GetFormalCharge() / 4.0,
        atom.GetTotalNumHs() / 4.0,
        float(atom.GetIsAromatic()),
        float(atom.IsInRing()),
        atom.GetMass() / 200.0
    ]
    return features

def bond_features(bond):
    return (
        one_hot_unknown(bond.GetBondType(), BOND_TYPES)
        + [
            float(bond.GetIsConjugated()),
            float(bond.IsInRing())
        ]
    )

ATOM_DIM = len(atom_features(Chem.MolFromSmiles("C").GetAtomWithIdx(0)))
BOND_DIM = len(bond_features(Chem.MolFromSmiles("CC").GetBondWithIdx(0)))

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    x = torch.tensor(
        [atom_features(atom) for atom in mol.GetAtoms()],
        dtype=torch.float32
    )

    edges = []
    edge_attrs = []

    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        feat = bond_features(bond)

        edges.extend([[i, j], [j, i]])
        edge_attrs.extend([feat, feat])

    if edges:
        edge_index = torch.tensor(
            edges, dtype=torch.long
        ).t().contiguous()

        edge_attr = torch.tensor(
            edge_attrs, dtype=torch.float32
        )
    else:
        edge_index = torch.empty(
            (2, 0), dtype=torch.long
        )
        edge_attr = torch.empty(
            (0, BOND_DIM), dtype=torch.float32
        )

    return Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr
    )

print("Atom features:", ATOM_DIM)
print("Bond features:", BOND_DIM)
# AttentiveFP
from torch_geometric.nn import AttentiveFP

class HybridAttentiveFP(nn.Module):
    def __init__(
        self,
        atom_dim,
        bond_dim,
        solvent_dim,
        hidden_dim=512,
        num_layers=3,
        num_timesteps=3,
        dropout=0.2
    ):
        super().__init__()

        # Molecular graph encoder
        self.graph_encoder = AttentiveFP(
            in_channels=atom_dim,
            hidden_channels=hidden_dim,
            out_channels=hidden_dim,
            edge_dim=bond_dim,
            num_layers=num_layers,
            num_timesteps=num_timesteps,
            dropout=dropout
        )

        # Solvent composition encoder
        self.solvent_encoder = nn.Sequential(
            nn.Linear(solvent_dim, 128),
            nn.ReLU(),
            nn.LayerNorm(128),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU()
        )

        # Combined regression head
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim + 64, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, data):

        # Graph-level molecular representation
        graph_embedding = self.graph_encoder(
            data.x,
            data.edge_index,
            data.edge_attr,
            data.batch
        )

        # Solvent representation
        solvent_embedding = self.solvent_encoder(
            data.solvent
        )

        # Combine molecular and solvent features
        combined = torch.cat(
            [graph_embedding, solvent_embedding],
            dim=1
        )

        return self.regressor(combined)




# MPNN
from torch_geometric.nn import NNConv, Set2Set
class HybridMPNN(nn.Module):
    def __init__(
        self,
        atom_dim,
        bond_dim,
        solvent_dim,
        hidden_dim=128,
        num_steps=3,
        set2set_steps=3,
        dropout=0.2
    ):
        super().__init__()

        self.num_steps = num_steps

        self.atom_encoder = nn.Linear(
            atom_dim, hidden_dim
        )

        # Bond-conditioned message function
        edge_network = nn.Sequential(
            nn.Linear(bond_dim, 128),
            nn.ReLU(),
            nn.Linear(128, hidden_dim * hidden_dim)
        )

        self.conv = NNConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            nn=edge_network,
            aggr="mean"
        )

        self.gru = nn.GRU(
            hidden_dim, hidden_dim
        )

        self.pool = Set2Set(
            hidden_dim,
            processing_steps=set2set_steps
        )

        self.solvent_encoder = nn.Sequential(
            nn.Linear(solvent_dim, 128),
            nn.ReLU(),
            nn.LayerNorm(128),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU()
        )

        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 64, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, data):
        x = F.relu(self.atom_encoder(data.x))

        h = x.unsqueeze(0)

        for _ in range(self.num_steps):
            m = F.relu(
                self.conv(x, data.edge_index, data.edge_attr)
            )

            x, h = self.gru(m.unsqueeze(0), h)
            x = x.squeeze(0)

        graph_embedding = self.pool(
            x, data.batch
        )

        solvent_embedding = self.solvent_encoder(
            data.solvent
        )

        combined = torch.cat(
            [graph_embedding, solvent_embedding],
            dim=1
        )

        return self.regressor(combined)




def canonicalize_smiles(smiles):
    mol = Chem.MolFromSmiles(str(smiles).strip())
    if mol is None or mol.GetNumAtoms() == 0:
        raise ValueError('SMILES is empty or invalid')
    return Chem.MolToSmiles(mol, canonical=True)

def normalize_ratios(names, text, allowed):
    if not names or len(names) != len(set(names)):
        raise ValueError('Select at least one solvent without duplicates')
    if set(names) - set(allowed):
        raise ValueError('The selected solvent is not supported by this model')
    parts = re.split(r'[:;\s]+', str(text).strip())
    try:
        values = np.array([float(x) for x in parts], dtype=np.float64)
    except ValueError:
        raise ValueError('Enter ratios such as 6:1 or 0.857:0.143')
    if len(values) != len(names):
        raise ValueError('Provide one ratio for each selected solvent')
    if not np.isfinite(values).all() or (values < 0).any() or not np.isfinite(values.sum()) or values.sum() <= 0:
        raise ValueError('Ratios must be finite, nonnegative, and have a positive total')
    return dict(zip(names, values / values.sum()))

class Predictor:
    def __init__(self, checkpoint_path, expected=None):
        self.path = Path(checkpoint_path)
        if not self.path.is_file():
            raise FileNotFoundError(f'Checkpoint not found: {self.path}')
        saved = torch.load(self.path, map_location='cpu', weights_only=True)
        self.name = saved['model_name']
        if expected is not None and self.name != expected:
            raise ValueError(f'Expected a {expected} checkpoint, received {self.name}')
        classes = {'AttentiveFP': HybridAttentiveFP, 'MPNN': HybridMPNN}
        self.solvents = list(saved['solvent_cols'])
        self.inputs = list(saved['input_solvent_cols'])
        if self.inputs != self.solvents + ['missing_mobile_phase']:
            raise ValueError('Unsupported checkpoint schema')
        config = saved['model_config']
        if (config['atom_dim'],config['bond_dim'],config['solvent_dim']) != (ATOM_DIM,BOND_DIM,len(self.inputs)):
            raise ValueError('Feature dimensions do not match the checkpoint')
        self.model = classes[self.name](**config).cpu().eval()
        self.model.load_state_dict(saved['model_state_dict'], strict=True)

    @torch.inference_mode()
    def predict(self, smiles, names, ratio_text):
        canonical = canonicalize_smiles(smiles)
        mixture = normalize_ratios(names, ratio_text, self.solvents)
        graph = smiles_to_graph(canonical)
        graph.solvent = torch.tensor([[mixture.get(s,0.) for s in self.solvents]+[0.]], dtype=torch.float32)
        value = float(self.model(Batch.from_data_list([graph])).item())
        return {'SMILES': canonical, 'Rf': value, 'mixture': mixture}


In [ ]:
predictor = Predictor(CHECKPOINT, expected=MODEL_NAME)
print('Loaded:', predictor.name, '| Solvents:', len(predictor.solvents))

In [ ]:
import ipywidgets as w
from IPython.display import display, clear_output
import html
import math

smiles_input = w.Textarea(
    description='SMILES:', placeholder='Enter SMILES',
    layout=w.Layout(width='95%'))

# Each solvent has its own ratio; no keyboard modifier is needed.
solvent_rows = []
for name in predictor.solvents:
    selected = w.Checkbox(value=False, description=name, indent=False,
                          layout=w.Layout(width='300px', min_width='300px', height='36px', flex='0 0 auto'))
    amount = w.Text(value='1', placeholder='Example: 6 or 0.857',
                    disabled=True, layout=w.Layout(width='180px', min_width='120px', height='36px', flex='0 0 auto'))
    solvent_rows.append((name, selected, amount))

order = w.HTML(value='No solvents selected.')

def collect_mixture():
    names, ratios = [], []
    for name, selected, amount in solvent_rows:
        if selected.value:
            names.append(name)
            text = amount.value.strip().replace(',', '.')
            try:
                number = float(text)
            except ValueError:
                raise ValueError(f'The ratio for {name} must be a number, such as 6 or 0.857')
            if not math.isfinite(number) or number < 0:
                raise ValueError(f'The ratio for {name} must be finite and nonnegative')
            ratios.append(str(number))
    ratio_text = ':'.join(ratios)
    # Use the same validation and normalization as the predictor.
    normalize_ratios(names, ratio_text, predictor.solvents)
    return names, ratio_text

def update_mixture(change=None):
    for name, selected, amount in solvent_rows:
        amount.disabled = not selected.value
    try:
        names, ratio_text = collect_mixture()
        mixture = normalize_ratios(names, ratio_text, predictor.solvents)
        order.value = '<b>Mixture:</b> ' + html.escape(' + '.join(
            f'{name}: {fraction:.2%}' for name, fraction in mixture.items()))
    except ValueError as exc:
        order.value = html.escape(str(exc))

for name, selected, amount in solvent_rows:
    selected.observe(update_mixture, names='value')
    amount.observe(update_mixture, names='value')

solvent_box = w.VBox(
    [w.HBox([selected, amount], layout=w.Layout(height='44px', min_height='44px', flex='0 0 auto', align_items='center')) for _, selected, amount in solvent_rows],
    layout=w.Layout(width='95%', height='400px', min_height='400px', overflow='auto'))
button = w.Button(description='Predict Rf', button_style='success')
out = w.Output()

def on_predict(_):
    with out:
        clear_output()
        try:
            names, ratio_text = collect_mixture()
            result = predictor.predict(smiles_input.value, names, ratio_text)
            print('Canonical SMILES:', result['SMILES'])
            print(f"Predicted Rf: {result['Rf']:.4f}")
            display(pd.DataFrame(result['mixture'].items(),
                                 columns=['Solvent', 'Normalized fraction']))
        except Exception as exc:
            print('Error:', str(exc))

button.on_click(on_predict)
display(
    smiles_input,
    w.HTML('<b>Select solvents and enter a ratio next to each name.</b><br>'
           'Example: select n-Hexane and enter 6; select Ethyl acetate and enter 1. '
           'Use relative parts or percentages; values are normalized automatically.'),
    solvent_box, order, button, out)
